# Post-training quantization cho GPT mini tiếng Việt

Bạn lấy GPT mini đã pretrain ở module 05 (checkpoint `gpt-mini-v1` của bài Advanced 01) và lượng tử hoá trọng số **sau khi
train** (PTQ), không cần dữ liệu train hay gradient. Câu hỏi: mỗi bit bớt đi và mỗi cách chia scale (per-tensor,
per-channel, theo nhóm, đối xứng hay không) làm bpc trên validation tăng bao nhiêu, đổi lại tệp nhỏ đi bao nhiêu, và lớp
nào nhạy nhất.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) nạp checkpoint, đánh giá, vẽ; test bỏ qua chúng.
- Checkpoint: chép tệp `gpt-mini-v1` của bạn vào `data/raw/gpt-mini-vi/gpt_mini_vi.pt` hoặc đặt biến môi trường
  `GPT_MINI_CKPT`. Chưa có thì notebook tự train một GPT nhỏ dự phòng (kết luận vẫn kiểm được, số tuyệt đối khác).
- Đây là **fake quantization**: trọng số được làm tròn về lưới lượng tử rồi lưu lại dạng float32. Nó đo đúng ảnh hưởng tới
  độ chính xác, không đo tốc độ (tốc độ cần kernel int thật, bài Advanced).
- Chấm: `uv run pytest tracks/ai-engineer/modules/10-quantization/exercises/intermediate/01-ptq-gpt-mini -q`

In [ ]:
# explore: giới hạn luồng, tìm gốc repo, kiểm Wikipedia tiếng Việt (tải một lần bằng env gốc), chỗ để checkpoint
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
WIKI = ROOT / "data/raw/wikipedia-vi"
if not (WIKI.is_dir() and any(WIKI.iterdir())):
    from data.download_dataset import ensure

    ensure("wikipedia-vi")
SHARD = WIKI / "20231101.vi/train-00002-of-00004.parquet"
CKPT = Path(os.environ.get("GPT_MINI_CKPT", ROOT / "data/raw/gpt-mini-vi/gpt_mini_vi.pt"))
FALLBACK = Path(os.environ.get("GPT_MINI_FALLBACK", ROOT / "data/raw/gpt-mini-vi/fallback_small.pt"))
print("checkpoint:", CKPT, "có" if CKPT.exists() else "chưa có")

In [ ]:
import copy
import math

import torch
import torch.nn as nn

## 1. Lượng tử hoá một ma trận trọng số

`fake_quantize_weight(w, bits, granularity="channel", symmetric=True)` nhận `w` shape `(out_features, in_features)` (như
`nn.Linear.weight`), trả tensor **cùng shape, cùng dtype** đã quantize rồi dequantize:

- `granularity="tensor"`: một scale cho cả ma trận; `"channel"`: một scale cho mỗi hàng (mỗi output channel); số nguyên
  `g`: một scale cho mỗi nhóm `g` cột liền nhau trên một hàng (như bài Basic 02; không chia hết thì `ValueError`).
- `symmetric=True`: absmax, mã $$[-q_{max}, q_{max}]$$ với $$q_{max} = 2^{b-1} - 1$$. `symmetric=False`: zero-point, mã
  $$[0, 2^b - 1]$$, khoảng chứa 0, zero-point làm tròn thành số nguyên. Khối (tensor, hàng, nhóm) toàn 0 dùng scale 1.
- Làm tròn bằng `torch.round`.

In [ ]:
def fake_quantize_weight(w: torch.Tensor, bits: int, granularity="channel", symmetric: bool = True) -> torch.Tensor:
    out_f, in_f = w.shape
    if granularity == "tensor":
        g = w.reshape(1, 1, -1)
    elif granularity == "channel":
        g = w.reshape(out_f, 1, in_f)
    else:
        size = int(granularity)
        if in_f % size:
            raise ValueError(f"in_features {in_f} không chia hết cho nhóm {size}")
        g = w.reshape(out_f, in_f // size, size)
    if symmetric:
        qmax = 2 ** (bits - 1) - 1
        scale = g.abs().amax(dim=-1, keepdim=True) / qmax
        scale = torch.where(scale == 0, torch.ones_like(scale), scale)
        out = torch.clamp(torch.round(g / scale), -qmax, qmax) * scale
    else:
        qmax = 2**bits - 1
        lo = g.amin(dim=-1, keepdim=True).clamp(max=0)
        hi = g.amax(dim=-1, keepdim=True).clamp(min=0)
        scale = (hi - lo) / qmax
        scale = torch.where(scale == 0, torch.ones_like(scale), scale)
        zero = torch.clamp(torch.round(-lo / scale), 0, qmax)
        out = (torch.clamp(torch.round(g / scale) + zero, 0, qmax) - zero) * scale
    return out.reshape(out_f, in_f)

## 2. Lượng tử hoá cả model, đếm byte

- `quantize_model(model, bits, granularity="channel", symmetric=True, include_embedding=False)`: trả **bản sao**
  (`copy.deepcopy`, model gốc không đổi) trong đó trọng số của mọi `nn.Linear` đã qua `fake_quantize_weight`. Bias,
  LayerNorm, `pos_emb` giữ nguyên. GPT mini dùng chung một tensor cho `lm_head.weight` và `tok_emb.weight` (weight tying):
  một `nn.Linear` có trọng số **là** trọng số của một `nn.Embedding` chỉ được quantize khi `include_embedding=True`, và khi đó
  quantize đúng một lần, giữ nguyên việc dùng chung (mỗi hàng là một token).
- `weight_storage_bytes(model, bits, granularity, symmetric=True, include_embedding=False)`: số byte để lưu model theo cấu
  hình đó. Mỗi tham số (tensor dùng chung đếm một lần): tensor được quantize tốn `numel * bits / 8` byte cộng 2 byte (fp16)
  cho mỗi scale, cộng thêm 1 byte cho mỗi zero-point nếu bất đối xứng; mọi tham số khác 4 byte mỗi phần tử (float32). Số scale:
  1 (tensor), `out_features` (channel), `out_features * in_features / g` (nhóm). Trả `float`.

In [ ]:
def _quantized_linears(model: nn.Module, include_embedding: bool) -> list[nn.Linear]:
    embedding_weights = {id(m.weight) for m in model.modules() if isinstance(m, nn.Embedding)}
    return [m for m in model.modules()
            if isinstance(m, nn.Linear) and (include_embedding or id(m.weight) not in embedding_weights)]


def quantize_model(model: nn.Module, bits: int, granularity="channel", symmetric: bool = True,
                   include_embedding: bool = False) -> nn.Module:
    q = copy.deepcopy(model)
    with torch.no_grad():
        for m in _quantized_linears(q, include_embedding):
            m.weight.copy_(fake_quantize_weight(m.weight, bits, granularity, symmetric))
    return q


def weight_storage_bytes(model: nn.Module, bits: int, granularity="channel", symmetric: bool = True,
                         include_embedding: bool = False) -> float:
    quantized = {id(m.weight): m.weight for m in _quantized_linears(model, include_embedding)}
    total = 0.0
    for p in model.parameters():
        if id(p) in quantized:
            out_f, in_f = p.shape
            n_scales = 1 if granularity == "tensor" else out_f if granularity == "channel" else out_f * in_f // int(granularity)
            total += p.numel() * bits / 8 + n_scales * (2 if symmetric else 3)
        else:
            total += 4 * p.numel()
    return total

## 3. Lớp nào nhạy nhất

`layer_sensitivity(model, names, quantize_fn, eval_fn)`: với mỗi tên trong `names` (tên module `nn.Linear`, như
`"blocks.0.attn.qkv"`), tạo một bản sao của model trong đó **chỉ** trọng số của lớp đó được thay bằng `quantize_fn(weight)`,
đo `eval_fn(bản sao)`; trả `dict` theo đúng thứ tự `names`, giá trị là độ chênh so với `eval_fn(model)` gốc (đo một lần).
Model gốc không đổi. Đây là cách tìm lớp cần giữ nhiều bit hơn (mixed precision).

In [ ]:
def layer_sensitivity(model: nn.Module, names: list[str], quantize_fn, eval_fn) -> dict[str, float]:
    base = eval_fn(model)
    out = {}
    for name in names:
        q = copy.deepcopy(model)
        layer = q.get_submodule(name)
        with torch.no_grad():
            layer.weight.copy_(quantize_fn(layer.weight))
        out[name] = eval_fn(q) - base
    return out

## 4. Thí nghiệm

Các cell dưới đây không được chấm. Trên CPU mỗi lần đánh giá mất vài giây tới vài chục giây (tuỳ cỡ checkpoint), cả mục này
khoảng 5-10 phút; trên GPU (kernel `ai-gpu`) đánh giá đủ 100 bài validation, nhanh hơn nhiều.

In [ ]:
# explore: nạp lớp GPT (Basic 01) và các hàm dữ liệu, đánh giá (Intermediate 01) của module 05: bài của bạn nếu đã làm, không thì lời giải
import importlib.util
import json
import types

M05 = ROOT / "tracks/ai-engineer/modules/05-transformers-llm-architecture"


def first_working(loaders, probe):
    for load in loaders:
        module = load()
        try:
            probe(module)
            return module
        except NotImplementedError:
            continue
    raise RuntimeError("không nạp được code module 05")


def py_loader(path):
    def load():
        spec = importlib.util.spec_from_file_location("gpt_mini", path)
        module = importlib.util.module_from_spec(spec)
        sys.modules["gpt_mini"] = module
        spec.loader.exec_module(module)
        return module
    return load


def nb_loader(path):
    def load():
        module = types.ModuleType("m05_train_gpt")
        cells = json.loads(path.read_text(encoding="utf-8"))["cells"]
        for cell in cells:
            if cell["cell_type"] == "code" and "explore" not in cell.get("metadata", {}).get("tags", []):
                exec("".join(cell["source"]), module.__dict__)
        module.__file__ = str(path)
        return module
    return load


gpt_mini = first_working(
    [py_loader(M05 / "exercises/basic/01-gpt-mini/starter/gpt.py"), py_loader(M05 / "solutions/basic/01-gpt-mini/gpt.py")],
    lambda m: m.GPT(m.GPTConfig(vocab_size=10, block_size=4, n_layer=1, n_head=1, n_embd=8))(torch.zeros(1, 2, dtype=torch.long)),
)
m05 = first_working(
    [nb_loader(M05 / "exercises/intermediate/01-train-gpt-wikipedia/starter/train_gpt.ipynb"),
     nb_loader(M05 / "solutions/intermediate/01-train-gpt-wikipedia/train_gpt.ipynb")],
    lambda m: (m.sliding_windows(10, 4, 2), m.split_articles(["a", "b", "c"], 1, 1, 0), m.nll_to_metrics(1.0, 1, 1),
               m.lr_at(0, 1e-3, 10, 100, 1e-4)),
)
print("GPT:", gpt_mini.__file__)
print("hàm đánh giá:", m05.__file__)

In [ ]:
# explore: nạp checkpoint gpt-mini-v1 của bạn; chưa có thì train một GPT nhỏ dự phòng (CPU khoảng 4 phút, lưu lại để dùng tiếp)
import dataclasses
import time

from tokenizers import Tokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(4)
articles = m05.long_articles(SHARD, 2000)
train_articles, val_articles = m05.split_articles(articles, 1000, 100, seed=0)


def load_gpt_mini(path):
    raw = torch.load(path, map_location="cpu", weights_only=True)
    if raw.get("format") != "gpt-mini-v1":
        raise ValueError(f"{path} không phải định dạng gpt-mini-v1")
    model = gpt_mini.GPT(gpt_mini.GPTConfig(**raw["config"]))
    model.load_state_dict(raw["state_dict"])
    return model.eval(), Tokenizer.from_str(raw["tokenizer"])


def train_fallback(path, steps=600):
    tok = m05.train_bpe("\n\n".join(train_articles), 8000)
    ids = torch.tensor(tok.encode("\n\n".join(train_articles)).ids, dtype=torch.long)
    torch.manual_seed(0)
    config = gpt_mini.GPTConfig(vocab_size=8000, block_size=128, n_layer=4, n_head=4, n_embd=128)
    model = gpt_mini.GPT(config)
    opt = m05.configure_optimizer(model, 1e-3, 0.1)
    g = torch.Generator().manual_seed(0)
    for step in range(steps):
        for group in opt.param_groups:
            group["lr"] = m05.lr_at(step, 1e-3, 100, steps, 1e-4)
        x, y = m05.get_batch(ids, 32, 128, g)
        _, loss = model(x, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"format": "gpt-mini-v1", "config": dataclasses.asdict(config),
                "state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
                "tokenizer": tok.to_str(), "step": steps, "metrics": {}}, path)


source = CKPT if CKPT.exists() else FALLBACK
if not source.exists():
    print("chưa có checkpoint module 05: train GPT nhỏ dự phòng...")
    start = time.perf_counter()
    train_fallback(FALLBACK)
    print(f"xong sau {time.perf_counter() - start:.0f} s, lưu ở {FALLBACK}")
model, tok = load_gpt_mini(source)
BLOCK = model.config.block_size
print(source.name, model.config, f"{model.num_params():,} tham số")

In [ ]:
# explore: validation của module 04/05; CPU chỉ lấy 20 bài đầu cho nhanh, GPU lấy đủ 100 bài
N_VAL = 100 if DEVICE == "cuda" else 20
val_text = "\n\n".join(val_articles[:N_VAL])
val_ids = torch.tensor(tok.encode(val_text).ids, dtype=torch.long)


def val_bpc(m) -> float:
    total, n = m05.evaluate_nll(m.to(DEVICE), val_ids, BLOCK, BLOCK // 2, device=DEVICE)
    return m05.nll_to_metrics(total, n, len(val_text))["bpc"]


start = time.perf_counter()
BASE_BPC = val_bpc(model)
print(f"{N_VAL} bài, {len(val_text):,} ký tự, {len(val_ids):,} token | float32 bpc {BASE_BPC:.4f}"
      f" ({time.perf_counter() - start:.1f} s mỗi lần đánh giá trên {DEVICE})")

In [ ]:
# explore: bảng PTQ: bpc và dung lượng theo số bit, mức chia scale, đối xứng hay không
configs = [
    (8, "tensor", True), (8, "channel", True),
    (4, "tensor", True), (4, "channel", True), (4, "channel", False), (4, 128, True), (4, 32, True), (4, 32, False),
    (3, "channel", True), (3, 32, True), (3, 32, False),
    (2, 32, True), (2, 32, False),
]
rows = [("float32", "", "", BASE_BPC, sum(4 * p.numel() for p in model.parameters()) / 1e6)]
for bits, gran, sym in configs:
    if isinstance(gran, int) and model.config.n_embd % gran:
        continue
    b = val_bpc(quantize_model(model, bits, gran, sym))
    rows.append((f"int{bits}", gran, "đối xứng" if sym else "zero-point", b, weight_storage_bytes(model, bits, gran, sym) / 1e6))
print(f"{'cấu hình':10s} {'scale':>8s} {'kiểu':>11s} {'bpc':>8s} {'Δbpc':>8s} {'MB':>7s}")
for name, gran, kind, b, mb in rows:
    print(f"{name:10s} {str(gran):>8s} {kind:>11s} {b:8.4f} {b - BASE_BPC:+8.4f} {mb:7.2f}")

In [ ]:
# explore: có quantize luôn embedding/lm_head (dùng chung) không
for bits, gran in [(8, "channel"), (4, "channel"), (4, 32)]:
    if isinstance(gran, int) and model.config.n_embd % gran:
        continue
    without = val_bpc(quantize_model(model, bits, gran))
    with_emb = val_bpc(quantize_model(model, bits, gran, include_embedding=True))
    print(f"int{bits} {gran}: Linear thôi {without:.4f} | cả embedding {with_emb:.4f}"
          f" | MB {weight_storage_bytes(model, bits, gran) / 1e6:.2f} -> {weight_storage_bytes(model, bits, gran, include_embedding=True) / 1e6:.2f}")

In [ ]:
# explore: phân phối trọng số: tỉ lệ max|w| / std theo lớp (ngoại lai làm hỏng per-tensor)
import matplotlib.pyplot as plt

linear_names = [n for n, m in model.named_modules() if isinstance(m, nn.Linear) and m.weight is not model.tok_emb.weight]
ratios = {n: (model.get_submodule(n).weight.abs().max() / model.get_submodule(n).weight.std()).item() for n in linear_names}
for n, r in ratios.items():
    print(f"{n:22s} max|w|/std = {r:5.1f}")
w = model.get_submodule(max(ratios, key=ratios.get)).weight.detach().flatten().cpu()
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].hist(w.numpy(), bins=200, log=True)
ax[0].set_title(f"{max(ratios, key=ratios.get)}: histogram (thang log)")
row_absmax = model.get_submodule(max(ratios, key=ratios.get)).weight.detach().abs().amax(dim=1).cpu()
ax[1].plot(row_absmax.numpy(), lw=0.7)
ax[1].set_title("max|w| theo từng hàng (output channel)")
plt.tight_layout(); plt.show()

In [ ]:
# explore: độ nhạy từng lớp với int3 per-channel, rồi mixed precision: giữ các lớp nhạy nhất ở int8
sens = layer_sensitivity(model, linear_names, lambda w: fake_quantize_weight(w, 3, "channel"), val_bpc)
ranked = sorted(sens, key=sens.get, reverse=True)
for n in ranked:
    print(f"{n:22s} Δbpc {sens[n]:+.4f}")
print("tổng các Δ riêng lẻ:", f"{sum(sens.values()):+.4f}", "| quantize tất cả cùng lúc:", f"{val_bpc(quantize_model(model, 3, 'channel')) - BASE_BPC:+.4f}")
for k in [0, 2, 4]:
    q = quantize_model(model, 3, "channel")
    with torch.no_grad():
        for n in ranked[:k]:
            q.get_submodule(n).weight.copy_(fake_quantize_weight(model.get_submodule(n).weight, 8, "channel"))
    print(f"giữ {k} lớp nhạy nhất ở int8: Δbpc {val_bpc(q) - BASE_BPC:+.4f}")

## 5. Đọc kết quả

So với mục 2 và 4 phần lý thuyết. Trả lời bằng số của chính bạn (con số phụ thuộc checkpoint):

1. Ở 8 bit, mức chia scale có quan trọng không? Ở 4 bit thì sao? So per-tensor, per-channel và nhóm 32, rồi đối chiếu với tỉ
   lệ `max|w|/std` của từng lớp để giải thích.
2. Ở 3 và 2 bit, so đối xứng với zero-point cùng mức chia. Đếm xem mỗi cách thực sự dùng bao nhiêu mã, rồi giải thích độ chênh.
3. Cột MB: tính bit/trọng số thực của int4 nhóm 32 và so với "4 bit" trên giấy.
4. Embedding/`lm_head` dùng chung là ma trận lớn nhất của một model nhỏ: quantize nó lợi bao nhiêu MB, mất bao nhiêu bpc?
5. Độ nhạy: lớp nào đứng đầu? Tổng các Δ riêng lẻ so với Δ khi quantize tất cả cùng lúc thế nào? Giữ vài lớp ở int8 lấy lại
   được bao nhiêu?

Sau khi chạy hết notebook, làm phần **trắc nghiệm** của bài trong app.